In [0]:
%load_ext autoreload
%autoreload 2

import pandas as pd, numpy as np
from pyspark.sql import functions as F
import formulaic as frm, sklearn as sk, glum as glm
import xgboost as xgb
from sklearn.metrics import d2_tweedie_score
import tempfile, os, mlflow, joblib


In [0]:
def safe_get(var_name, default):
    try:
        return dbutils.widgets.get(var_name)
    except KeyError:
        return default

In [0]:
catalog = safe_get("catalog", "workspace")
schema = safe_get("schema", "mlops_dev")

tbl_ilec_data = (
    spark.read.table(f"{catalog}.{schema}.ilec_data")
    .filter(
        (F.trim(F.col("Insurance_Plan")) == "TERM") &
        (F.col("SOA_Post_Lvl_Ind") != F.lit("PLT")) &
        (F.col("ExpDth_VBT2015wMI_Cnt") > F.lit(0.0))
    )
    .withColumn("DATASET",
        F.when(F.col("Observation_Year") <= F.lit(2017), "TRAIN")
         .otherwise("TEST")
    )
)

tbl_ilec_data.count()

# COMMAND ----------

def agg_data(df : F.DataFrame) -> F.DataFrame:
    return (
        df
        .groupBy(["Observation_Year", "Issue_Year", "Issue_Age", "Sex", "Smoker_Status", "Attained_Age", "Face_Amount_Band"])
        .agg(
            F.sum("Death_Count").alias("Death_Count"),
            F.sum("ExpDth_VBT2015wMI_Cnt").alias("ExpDth_VBT2015wMI_Cnt")
        )
    )

tbl_train = (
    tbl_ilec_data
    .filter(F.col("DATASET") == F.lit("TRAIN"))
)

tbl_test = (
    tbl_ilec_data
    .filter(F.col("DATASET") == F.lit("TEST"))
)

df_train = agg_data(tbl_train).toPandas()
df_test = agg_data(tbl_test).toPandas()


In [0]:
x_mat_formula = frm.Formula(" ~ cr(Attained_Age, df=4, lower_bound=18, upper_bound=90 )*Smoker_Status*Sex + Face_Amount_Band - 1")

In [0]:
X_train = x_mat_formula.get_model_matrix(df_train)
offset_train = np.log(df_train["ExpDth_VBT2015wMI_Cnt"])
y_train = df_train["Death_Count"]

X_val = x_mat_formula.get_model_matrix(df_test)
offset_val = np.log(df_test["ExpDth_VBT2015wMI_Cnt"])
y_val = df_test["Death_Count"]

In [0]:
df_train.attrs = {}
df_train.to_parquet("df_train.parquet")

In [0]:
coef_table = glmnet.coef_table()

term_groups = [""] * coef_table.shape[0]

def slice_to_indices(s: slice, length: int) -> range:
    return range(*s.indices(length))

term_groups[0] = "intercept"
for term, term_slice in X_train.model_spec.term_slices.items():
    for i in slice_to_indices(term_slice, coef_table.shape[0]):
        term_groups[i + 1] = term
    
term_groups


In [0]:
(
    pd.DataFrame({
        "term_value" : glmnet.coef_table(),
        "term_group" : term_groups
    })
    .reset_index(drop=False)
    .rename({"index":"term_name"}, axis=1)
).to_csv("sample_mapping.csv", index=False)


In [0]:
glmnet = glm.GeneralizedLinearRegressor(
    family="poisson",
    alpha_search=True,
    min_alpha_ratio=1e-6
)
glmnet.fit(
    X_train,
    y = y_train,
    offset = offset_train
)

In [0]:
df_train["ExpDth_VBT2015wMI_Cnt"].min()

In [0]:
from glm_tools import PoissonDecisionTree

dtree = PoissonDecisionTree("Death_Count", "ExpDth_VBT2015wMI_Cnt")
dtree.fit(df_train)
print(dtree)

In [0]:
df_train["model_pred"] = glmnet.predict(X_train, offset=offset_train)
dtree2 = PoissonDecisionTree("Death_Count", "model_pred")
dtree2.fit(df_train.drop("ExpDth_VBT2015wMI_Cnt", axis=1))
print(dtree2)

In [0]:
pd.DataFrame(df_train).to_parquet("test.parquet")

In [0]:
X_train.model_spec.factor_variables

In [0]:
X_train.model_spec.factor_terms

In [0]:
X_train.model_spec.variable_terms

In [0]:
X_train.model_spec.variables_by_source

In [0]:
X_train.model_spec.term_factors

In [0]:
X_train.model_spec.column_names

In [0]:
X_train.model_spec.term_slices["Sex"].

In [0]:
glmnet.coef_table()

In [0]:
X_train.model_spec.term_slices

In [0]:
X_train.model_spec.column_names

In [0]:
str_cols = list(map(
    lambda c: str(c),
    df_train.select_dtypes(
    include=['object', 'string']).columns
))

response_cols = [
    "Death_Count",
    "ExpDth_VBT2015wMI_Cnt"
]

num_cols = list(set(df_train.columns).difference(set(str_cols)).difference(set(response_cols)))

xfrm = [
    (
        "ohe",
        sk.preprocessing.OneHotEncoder(
            drop="first", handle_unknown="ignore",
        ),
        str_cols
    ),
    (
        "num", "passthrough", num_cols
    )
]

preproc = sk.compose.ColumnTransformer(
    transformers=xfrm,
    remainder="drop",
    verbose_feature_names_out=True
)
    
xgb_train = preproc.fit_transform(df_train)
xgb_offset = np.log(glmnet.predict(X_train, offset=offset_train))

In [0]:
df_train["Death_Count"].to_numpy('float32')

In [0]:
df_xgb_train = pd.DataFrame(preproc.fit_transform(df_train))
df_xgb_train.columns = preproc.get_feature_names_out().tolist()

In [0]:
xgb_mat = xgb.DMatrix(
    data=xgb_train,
    label=y_train,
    base_margin=xgb_offset, feature_names=preproc.get_feature_names_out().tolist()
  
)

params = {
    'objective': 'count:poisson',
    'max_depth': 3,
    'tree_method': 'exact',
    'grow_policy': 'lossguide',
    'seed':0
}

bst = xgb.train(
    params=params,
    dtrain=xgb_mat,
    num_boost_round=1
)


In [0]:
import json
from glm_tools import PoissonDecisionTree
# -------------------------------------------------
# 2) Parse tree JSON
# -------------------------------------------------
tree_json = json.loads(bst.get_dump(dump_format="json")[0])

# -------------------------------------------------
# 4) Compute split-level deviance table
# -------------------------------------------------
split_stats = collect_split_stats_pretty_path(tree_json, df_xgb_train, y_train, np.exp(xgb_offset))
split_df = pd.DataFrame(split_stats)

pretty_print_split_stats(split_df)

# print(split_df[[
#     "nodeid", "feature", "threshold",
#     "dev_parent", "dev_children", "dev_reduction",
#     "xgb_gain"
# ]])